# 01 · Teammate qualifying gaps

Teammates share (nearly) the same car, so their qualifying gap is the cleanest early signal of driver pace.

**Method:** for each race, the two teammates are compared in the last qualifying segment they *both* set a time in (Q3 if both reached it, else Q2, else Q1). Gap is in % of the teammate's lap: negative = faster. Gaps above the outlier threshold are flagged and excluded from season summaries.

**Known limitation:** teammates' cars are not identical (setups, upgrade timing, reliability), so this is an approximation.

Run `00_ingest_quali.ipynb` first.

In [8]:
# @title Setup: mount Drive, get the latest code, install FastF1
REPO_URL = "https://github.com/khushi075/f1-race-sim.git"  # @param {type:"string"}
DATA_DIR = "/content/drive/MyDrive/f1sim_data"  # @param {type:"string"}

import os, sys
from google.colab import drive

# Python 3.12+ compatibility patch for legacy packages or magics still using 'imp'
try:
    import imp
except ImportError:
    import types, importlib
    imp = types.ModuleType('imp')
    imp.reload = importlib.reload
    sys.modules['imp'] = imp

drive.mount("/content/drive")

REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q "fastf1>=3.9,<4"

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ["F1SIM_DATA_DIR"] = DATA_DIR

%load_ext autoreload
%autoreload 2
print("Code:", REPO_DIR, "| Data:", DATA_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 802 bytes | 802.00 KiB/s, done.
From https://github.com/khushi075/f1-race-sim
   de89b4d..04c63fe  main       -> origin/main
Updating de89b4d..04c63fe
Fast-forward
 f1sim/features.py | 17 ++++++++++++-----
 1 file changed, 12 insertions(+), 5 deletions(-)
/content/f1-race-sim
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Code: /content/f1-race-sim | Data: /content/drive/MyDrive/f1sim_data


In [9]:
# @title Load data and compute gaps
OUTLIER_PCT = 2.0  # @param {type:"number"}

import plotly.express as px
from f1sim.paths import get_paths
from f1sim.ingest import load_quali
from f1sim.features import teammate_gaps, season_summary

quali = load_quali(get_paths())
gaps = teammate_gaps(quali, outlier_pct=OUTLIER_PCT)
summary = season_summary(gaps)

n_sessions = len(quali.drop_duplicates(["Year", "Round"]))
print(f"{n_sessions} sessions | {len(gaps) // 2} teammate comparisons | "
      f"{int(gaps['Outlier'].sum()) // 2} flagged as outliers")

173 sessions | 1684 teammate comparisons | 72 flagged as outliers


In [3]:
# @title Season head-to-head table
YEAR = 2025  # @param {type:"integer"}

summary.loc[summary["Year"] == YEAR, ["Team", "Driver", "Teammate", "Races", "H2H", "MedianGapPct"]]

,Team,Driver,Teammate,Races,H2H,MedianGapPct
174,Red Bull,VER,LAW,2,2-0,-1.105
175,Red Bull,VER,TSU,20,20-0,-0.681
176,Alpine F1 Team,GAS,DOO,6,5-1,-0.509
177,Alpine F1 Team,GAS,COL,18,13-5,-0.381
178,Mercedes,RUS,ANT,23,20-3,-0.365
179,Aston Martin,ALO,STR,23,23-0,-0.363
180,Ferrari,LEC,HAM,24,19-5,-0.245
181,Haas F1 Team,BEA,OCO,22,13-9,-0.198
182,RB F1 Team,HAD,TSU,2,1-1,-0.196
183,RB F1 Team,HAD,LAW,21,16-5,-0.165


In [10]:
# @title Median gap to teammate, by driver
season = summary[summary["Year"] == YEAR]
fig = px.bar(
    season.sort_values("MedianGapPct"),
    x="MedianGapPct", y="Driver", color="Team", orientation="h",
    hover_data=["Teammate", "H2H", "Races"],
    labels={"MedianGapPct": "Median gap to teammate (%)"},
    title=f"{YEAR}: median qualifying gap to teammate (negative = faster)",
)
fig.add_vline(x=0, line_dash="dot")
fig.update_layout(height=650, yaxis={"categoryorder": "total descending"})
fig.show()

In [11]:
# @title One team, race by race
TEAM = "McLaren"  # @param {type:"string"}

teams = sorted(gaps.loc[gaps["Year"] == YEAR, "Team"].unique())
if TEAM not in teams:
    print(f"'{TEAM}' not found in {YEAR}. Pick one of:", teams)
else:
    tg = gaps[(gaps["Year"] == YEAR) & (gaps["Team"] == TEAM)]
    fig = px.line(
        tg, x="Round", y="GapPct", color="Driver", markers=True, symbol="Outlier",
        hover_data=["Event", "Teammate", "Segment"],
        labels={"GapPct": "Gap to teammate (%)"},
        title=f"{TEAM} {YEAR}: qualifying gap by round",
    )
    fig.add_hline(y=0, line_dash="dot")
    fig.show()

In [12]:
# @title One driver across seasons
DRIVER = "ALO"  # @param {type:"string"}

d = summary[summary["Driver"] == DRIVER].copy()
if d.empty:
    print(f"No data for '{DRIVER}'. Use the 3-letter code, e.g. HAM, VER, LEC.")
else:
    fig = px.line(
        d, x="Year", y="MedianGapPct", markers=True, text="Teammate",
        hover_data=["Team", "H2H", "Races"],
        labels={"MedianGapPct": "Median gap to teammate (%)"},
        title=f"{DRIVER}: median qualifying gap to teammate by season",
    )
    fig.update_traces(textposition="top center")
    fig.add_hline(y=0, line_dash="dot")
    fig.show()

In [14]:
# @title Inspect flagged outliers for the season
gaps.loc[gaps["Outlier"] & (gaps["Year"] == YEAR), ["Round", "Event", "Team", "Driver", "Teammate", "Segment", "GapPct"]]

,Round,Event,Team,Driver,Teammate,Segment,GapPct
3312,22,Las Vegas Grand Prix,Ferrari,HAM,LEC,Q1,2.004111
3313,22,Las Vegas Grand Prix,Ferrari,LEC,HAM,Q1,-1.964736
3318,22,Las Vegas Grand Prix,Mercedes,ANT,RUS,Q1,2.801739
3319,22,Las Vegas Grand Prix,Mercedes,RUS,ANT,Q1,-2.725381
3322,22,Las Vegas Grand Prix,Red Bull,TSU,VER,Q1,2.943821
3323,22,Las Vegas Grand Prix,Red Bull,VER,TSU,Q1,-2.859638


In [15]:
#The other missing Red Bull race which isn't in outlier, since the teammates never set time in same segment, finding out the round for that
vr = gaps[(gaps["Year"] == 2025) & (gaps["Driver"] == "VER")]
sorted(set(range(1, 25)) - set(vr["Round"]))

[7]